# Flughafen MUC - Redis Aufgaben

Wir arbeiten mit der Abflugtafel des Flughafens München: **25 Flüge** mit Airlines aus Europa, Amerika, Asien und dem Nahen Osten.

### Status-Verteilung im Datensatz
- 16 Flüge **ON_TIME**
- 3 Flüge **BOARDING**
- 4 Flüge **DELAYED**
- 1 Flug **DEPARTED**
- 2 Flüge **CANCELLED**

---
# Setup

> **Hinweis für Colab:** Wir installieren Redis in der Colab-VM und starten den Server als Hintergrundprozess. Pro Session einmal ausführen.

In [ ]:
# Redis-Server installieren und starten
!apt-get install -y redis-server > /dev/null
!redis-server --daemonize yes --save "" --appendonly no
print("Redis läuft auf localhost:6379")

Redis läuft auf localhost:6379


In [ ]:
!pip install -q redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 7.8 MB/s eta 0:00:00


In [ ]:
import redis
import time
import json
from datetime import datetime

## Verbindung zu Redis

In [ ]:
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
print("PING:", r.ping())

r.flushdb()
print("Datenbank ist leer und bereit.")

PING: True
Datenbank ist leer und bereit.


## Daten laden

Die Flugdaten sind direkt im Notebook. Die folgende Zelle ausführen, um sie in Redis zu laden.

In [ ]:
FLIGHTS = {
    "LH401":  {"airline": "Lufthansa",       "destination": "Frankfurt (FRA)",     "scheduled": "14:30", "gate": "B12", "status": "ON_TIME",   "delay_min": 0},
    "LH723":  {"airline": "Lufthansa",       "destination": "New York (JFK)",      "scheduled": "15:45", "gate": "H08", "status": "BOARDING",  "delay_min": 0},
    "EW132":  {"airline": "Eurowings",       "destination": "Hamburg (HAM)",       "scheduled": "14:50", "gate": "D04", "status": "DELAYED",   "delay_min": 15},
    "LH452":  {"airline": "Lufthansa",       "destination": "Tokio (NRT)",         "scheduled": "16:20", "gate": "H22", "status": "ON_TIME",   "delay_min": 0},
    "OS112":  {"airline": "Austrian",        "destination": "Wien (VIE)",          "scheduled": "14:55", "gate": "C18", "status": "ON_TIME",   "delay_min": 0},
    "AF1023": {"airline": "Air France",      "destination": "Paris (CDG)",         "scheduled": "15:30", "gate": "G14", "status": "CANCELLED", "delay_min": 0},
    "KL1804": {"airline": "KLM",             "destination": "Amsterdam (AMS)",     "scheduled": "15:10", "gate": "A06", "status": "BOARDING",  "delay_min": 0},
    "BA947":  {"airline": "British Airways", "destination": "London (LHR)",        "scheduled": "15:25", "gate": "A12", "status": "ON_TIME",   "delay_min": 0},
    "EW782":  {"airline": "Eurowings",       "destination": "Stuttgart (STR)",     "scheduled": "16:00", "gate": "D11", "status": "ON_TIME",   "delay_min": 0},
    "LH640":  {"airline": "Lufthansa",       "destination": "Dubai (DXB)",         "scheduled": "17:45", "gate": "H04", "status": "ON_TIME",   "delay_min": 0},
    "TK1632": {"airline": "Turkish",         "destination": "Istanbul (IST)",      "scheduled": "18:15", "gate": "B22", "status": "DELAYED",   "delay_min": 30},
    "EI354":  {"airline": "Aer Lingus",      "destination": "Dublin (DUB)",        "scheduled": "14:40", "gate": "A08", "status": "BOARDING",  "delay_min": 0},
    "LX1149": {"airline": "Swiss",           "destination": "Zürich (ZRH)",        "scheduled": "15:20", "gate": "C09", "status": "ON_TIME",   "delay_min": 0},
    "SQ327":  {"airline": "Singapore",       "destination": "Singapur (SIN)",      "scheduled": "22:30", "gate": "H18", "status": "ON_TIME",   "delay_min": 0},
    "AA71":   {"airline": "American",        "destination": "Chicago (ORD)",       "scheduled": "15:55", "gate": "G02", "status": "ON_TIME",   "delay_min": 0},
    "LH202":  {"airline": "Lufthansa",       "destination": "Berlin (BER)",        "scheduled": "14:35", "gate": "D15", "status": "DEPARTED",  "delay_min": 0},
    "IB3185": {"airline": "Iberia",          "destination": "Madrid (MAD)",        "scheduled": "16:05", "gate": "B07", "status": "ON_TIME",   "delay_min": 0},
    "AZ431":  {"airline": "ITA Airways",     "destination": "Rom (FCO)",           "scheduled": "15:50", "gate": "C04", "status": "DELAYED",   "delay_min": 45},
    "QR58":   {"airline": "Qatar",           "destination": "Doha (DOH)",          "scheduled": "21:00", "gate": "H10", "status": "ON_TIME",   "delay_min": 0},
    "DL83":   {"airline": "Delta",           "destination": "Atlanta (ATL)",       "scheduled": "16:30", "gate": "G06", "status": "ON_TIME",   "delay_min": 0},
    "UA907":  {"airline": "United",          "destination": "San Francisco (SFO)", "scheduled": "17:10", "gate": "G11", "status": "DELAYED",   "delay_min": 20},
    "LH1234": {"airline": "Lufthansa",       "destination": "Hannover (HAJ)",      "scheduled": "18:00", "gate": "D08", "status": "ON_TIME",   "delay_min": 0},
    "EK53":   {"airline": "Emirates",        "destination": "Dubai (DXB)",         "scheduled": "22:15", "gate": "H14", "status": "ON_TIME",   "delay_min": 0},
    "LH18":   {"airline": "Lufthansa",       "destination": "São Paulo (GRU)",     "scheduled": "22:00", "gate": "H20", "status": "CANCELLED", "delay_min": 0},
    "W64321": {"airline": "Wizz Air",        "destination": "Warschau (WAW)",      "scheduled": "19:30", "gate": "B05", "status": "ON_TIME",   "delay_min": 0},
}

print(f"{len(FLIGHTS)} Flüge im Datensatz.")

25 Flüge im Datensatz.


In [ ]:
# Daten in Redis laden: jeder Flug als Hash unter dem Key 'flight:<nummer>'
for flight_no, info in FLIGHTS.items():
    r.hset(f"flight:{flight_no}", mapping=info)

# Zusätzlich: Index aller Flugnummern als Set
r.sadd("flights:all", *FLIGHTS.keys())

print(f"In Redis: {r.scard('flights:all')} Flüge geladen.")
print("Beispiel - Daten von LH401:")
print(r.hgetall("flight:LH401"))

In Redis: 25 Flüge geladen.
Beispiel - Daten von LH401:
{'airline': 'Lufthansa', 'destination': 'Frankfurt (FRA)', 'scheduled': '14:30', 'gate': 'B12', 'status': 'ON_TIME', 'delay_min': '0'}


---
# Aufgaben

---
## Aufgabe 1 — Status eines Flugs setzen und auslesen

**Szenario:** Die Crew von **AF1023** (Paris) entscheidet sich kurzfristig doch noch zu starten — der Flug soll von `CANCELLED` zurück auf `BOARDING` gesetzt werden.

**Aufgabe:**
1. Setze das Feld `status` im Hash `flight:AF1023` auf `"BOARDING"`.
2. Lies den neuen Wert wieder aus und gib ihn aus.

> Tipp: `r.hset(key, field, value)` und `r.hget(key, field)`

**Deine Lösung:**

In [ ]:
r.hset("flight:AF1023", "status", "BOARDING")

neuer_status = r.hget("flight:AF1023", "status")
print(f"Neuer Status von AF1023: {neuer_status}")

Neuer Status von AF1023: BOARDING


---
## Aufgabe 2 — Lufthansa-Flüge filtern

**Szenario:** Das Lufthansa-Servicepersonal will sehen, welche **Lufthansa-Flüge** noch nicht gestartet sind.

**Aufgabe:**
1. Hole **alle** Flugnummern aus dem Set `flights:all`.
2. Filtere die Flüge, deren `airline` `"Lufthansa"` ist und deren `status` **nicht** `"DEPARTED"` ist.
3. Gib für jeden Treffer Flugnummer, Ziel und Status aus (jede Zeile ein Flug).

> Tipp: `r.smembers("flights:all")` liefert alle Flugnummern. Mit `r.hmget(key, [feld1, feld2])` kannst du mehrere Felder eines Hashes gleichzeitig lesen.

**Deine Lösung:**

In [ ]:
for fn in r.smembers("flights:all"):
    airline, destination, status = r.hmget(f"flight:{fn}", ["airline", "destination", "status"])
    if airline == "Lufthansa" and status != "DEPARTED":
        print(f"{fn:8s} {destination:22s} {status}")

LH452    Tokio (NRT)            ON_TIME
LH723    New York (JFK)         BOARDING
LH640    Dubai (DXB)            ON_TIME
LH18     São Paulo (GRU)        CANCELLED
LH1234   Hannover (HAJ)         ON_TIME
LH401    Frankfurt (FRA)        ON_TIME


---
## Aufgabe 3 — Gate-Änderungs-Benachrichtigung mit TTL

**Szenario:** Flug **BA947** (London) wechselt kurzfristig vom Gate `A12` auf `A18`. Die Hinweismeldung soll nur **5 Sekunden** auf der Anzeigetafel stehen — danach soll Redis sie automatisch löschen.

**Aufgabe:**
1. Speichere unter dem Key `gate_change:BA947` den String `"Gate-Wechsel von A12 auf A18"` mit TTL = **5 Sekunden**.
2. Lies den Wert sofort und gib ihn samt aktueller TTL aus.
3. Warte **6 Sekunden**.
4. Lies den Wert **erneut** und gib auch jetzt Wert und TTL aus.

> Tipp: `r.set(key, value, ex=sekunden)` und `r.ttl(key)`.
> TTL-Returncodes: positive Zahl = Restzeit, `-2` = Key existiert nicht mehr.

**Deine Lösung:**

In [ ]:
r.set("gate_change:BA947", "Gate-Wechsel von A12 auf A18", ex=5)

print(f"Wert: {r.get('gate_change:BA947')}")
print(f"TTL : {r.ttl('gate_change:BA947')} s")

print("\nWarte 6 Sekunden ...")
time.sleep(6)

print(f"Wert: {r.get('gate_change:BA947')}")
print(f"TTL : {r.ttl('gate_change:BA947')} s")

Wert: Gate-Wechsel von A12 auf A18
TTL : 5 s

Warte 6 Sekunden ...
Wert: None
TTL : -2 s


---
## Aufgabe 4 — Page-View-Counter für die Detailseiten

**Szenario:** Pro Flug zählen wir die Aufrufe der Detailseite. Wir wollen simulieren, dass der Münchner Charterflug **LH723** (nach New York) gerade besonders populär ist.

**Aufgabe:**
1. Simuliere folgende Page-Views mit `r.incr(...)`:
   - **150-mal** Aufruf der Seite zu `LH723`
   - **40-mal** Aufruf zu `LH401`
   - **12-mal** Aufruf zu `EW132`
2. Lies anschließend die drei Counter aus und gib sie aus.

Die Keys sollen die Form `views:flight:<flugnummer>` haben.

> Tipp: `r.incr(key)` erhöht atomar um 1. Es gibt auch `r.incrby(key, amount)` für mehrere Schritte auf einmal.

**Deine Lösung:**

In [ ]:
# 150x LH723
for _ in range(150):
    r.incr("views:flight:LH723")

# 40x LH401 - hier mit incrby für Effizienz
r.incrby("views:flight:LH401", 40)

# 12x EW132
for _ in range(12):
    r.incr("views:flight:EW132")

# Ausgabe
print(f"Views LH723: {r.get('views:flight:LH723')}")
print(f"Views LH401: {r.get('views:flight:LH401')}")
print(f"Views EW132: {r.get('views:flight:EW132')}")

Views LH723: 150
Views LH401: 40
Views EW132: 12


---
## Aufgabe 5 — Abonnenten verwalten mit Sets

**Szenario:** Vielflieger können Flüge abonnieren, um bei Verspätungen sofort eine Push-Nachricht zu bekommen.

**Aufgabe:**
1. Lege folgende Abos an (Key-Schema: `subscribers:flight:<flugnummer>`):
   - `LH401`: User `user:42`, `user:99`, `user:108`
   - `LH723`: User `user:42`, `user:200`
   - `BA947`: User `user:99`, `user:42`, `user:300`
2. Gib die Anzahl der Abonnenten je Flug aus.
3. Ermittle und gib die User aus, die **alle drei** Flüge abonniert haben (Schnittmenge).

> Tipp: `r.sadd(key, *werte)`, `r.scard(key)`, `r.sinter(key1, key2, key3)`.

**Deine Lösung:**

In [ ]:
r.sadd("subscribers:flight:LH401", "user:42", "user:99", "user:108")
r.sadd("subscribers:flight:LH723", "user:42", "user:200")
r.sadd("subscribers:flight:BA947", "user:99", "user:42", "user:300")

for fn in ["LH401", "LH723", "BA947"]:
    print(f"{fn}: {r.scard(f'subscribers:flight:{fn}')} Abonnenten")

gemeinsam = r.sinter(
    "subscribers:flight:LH401",
    "subscribers:flight:LH723",
    "subscribers:flight:BA947"
)
print(f"\nUser mit allen drei Abos: {gemeinsam}")

LH401: 3 Abonnenten
LH723: 2 Abonnenten
BA947: 3 Abonnenten

User mit allen drei Abos: {'user:42'}


---
## Aufgabe 6 — Verspätungs-Ranking als Sorted Set

**Szenario:** Der Operations-Manager will eine Live-Rangliste der **am stärksten verspäteten Flüge**.

**Aufgabe:**
1. Gehe alle Flüge aus `flights:all` durch.
2. Wenn ein Flug einen `delay_min` größer als `0` hat, füge ihn ins Sorted Set `delays:ranking` ein — als Score nimmst du die Verspätung in Minuten.
3. Gib die **Top 3** Flüge mit der höchsten Verspätung aus (Flugnummer + Minuten).

> Tipp: `r.zadd(key, {member: score})`, `r.zrevrange(key, 0, 2, withscores=True)` für die Top 3.

**Deine Lösung:**

In [ ]:
for fn in r.smembers("flights:all"):
    delay = int(r.hget(f"flight:{fn}", "delay_min"))
    if delay > 0:
        r.zadd("delays:ranking", {fn: delay})

print("Top 3 verspätete Flüge:")
for fn, score in r.zrevrange("delays:ranking", 0, 2, withscores=True):
    print(f"  {fn}: +{int(score)} min")

Top 3 verspätete Flüge:
  AZ431: +45 min
  TK1632: +30 min
  UA907: +20 min


---
## Shutdown

In [ ]:
r.flushdb()
!redis-cli -p 6379 shutdown nosave 2>/dev/null

print("Aufgaben beendet, alles aufgeräumt.")

Aufgaben beendet, alles aufgeräumt.
